# AgriVision AI - official Kaggle GPU training

**Developer / Project Author:** Isis Saritha Swapin, studying in Class 8 at Syndesmos Public School, Parumala, Thiruvalla.

**Contributor:** Swapin Vidya.

**Publisher:** PeachBot AI.

This notebook runs the official reproducible training path only. It does not publish to Hugging Face.

## GPU verification

In [ ]:
!nvidia-smi

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("Physical GPUs:", tf.config.list_physical_devices("GPU"))
print("Mixed precision policy:", tf.keras.mixed_precision.global_policy().name)

## Clone and checkout

In [ ]:
%cd /kaggle/working
!rm -rf agrivision-ai-pi
!git clone https://github.com/swapins/agrivision-ai-pi.git
%cd /kaggle/working/agrivision-ai-pi
!git checkout feature/kaggle-gpu-training
!git log -1 --oneline

## Install dependencies

In [ ]:
!python -m pip install --upgrade pip
!pip install -r training/requirements.txt
!pip install -r requirements-dev.txt

## Repository tests

In [ ]:
!pytest -q

## Official GPU training

In [ ]:
!python training/kaggle_train.py

## Display generated training artifacts

In [ ]:
from pathlib import Path
import json

output_dir = Path("training/output/agrivision-mobilenetv2")
history = json.loads((output_dir / "training_history.json").read_text())
manifest = json.loads((output_dir / "model_manifest.json").read_text())

print("training_history.json")
print(json.dumps(history, indent=2)[:4000])
print("\nmodel_manifest.json")
print(json.dumps(manifest, indent=2)[:4000])

## Final FLOAT metrics

In [ ]:
float_metrics = manifest["float_test_metrics"]
print("accuracy:", float_metrics["accuracy"])
print("balanced accuracy:", float_metrics["balanced_accuracy"])
print("macro-F1:", float_metrics["macro_f1"])
print("confusion matrix:")
print(float_metrics["confusion_matrix"])
print("per-class precision/recall/F1:")
for label in manifest["labels"]:
    row = float_metrics["classification_report"][label]
    print(label, {k: row[k] for k in ("precision", "recall", "f1-score")})

## Final INT8 metrics and release gate

In [ ]:
int8_metrics = manifest["int8_test_metrics"]
gate = manifest["release_gate"]
print("accuracy:", int8_metrics["accuracy"])
print("balanced accuracy:", int8_metrics["balanced_accuracy"])
print("macro-F1:", int8_metrics["macro_f1"])
print("confusion matrix:")
print(int8_metrics["confusion_matrix"])
print("prediction distribution:", int8_metrics["prediction_counts"])
print("collapsed_prediction:", int8_metrics["collapsed_prediction"])

passes_gate = (
    int8_metrics["accuracy"] >= gate["min_int8_accuracy"]
    and int8_metrics["balanced_accuracy"] >= gate["min_int8_balanced_accuracy"]
    and int8_metrics["macro_f1"] >= gate["min_int8_macro_f1"]
    and int8_metrics["classes_predicted"] == len(manifest["labels"])
)
print("PASS RELEASE GATE" if passes_gate else "FAIL RELEASE GATE")

## Package artifacts

In [ ]:
!zip -r agrivision-kaggle-training-artifacts.zip training/output/agrivision-mobilenetv2
!ls -lh agrivision-kaggle-training-artifacts.zip